In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0
!pip install -q -U transformers


In [ ]:
import os
import sys
import time
import traceback
import torch
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working"
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
def resolve(local_dir, hf_id):
    return local_dir if os.path.isdir(local_dir) else hf_id
SMALL = resolve("/kaggle/input/qwen35-08b", "Qwen/Qwen3.5-0.8B")
LARGE = resolve("/kaggle/input/qwen35-4b", "Qwen/Qwen3.5-4B")


In [ ]:
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
    Returns:
        J: [|U|, D] Jacobian matrix (unnormalized).
    """
    model.eval()
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    # Match model dtype to avoid mismatch in decoder (fp16 models on GPU)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
    J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    ponytail: copy.deepcopy works but is slow. Manual clone for speed + safety.
    """
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            new_k = [k.clone() if k is not None else None for k in layer.keys]
            new_v = [v.clone() if v is not None else None for v in layer.values]
            new_cache.layers.append(DynamicLayer(keys=new_k, values=new_v))
        else:
            new_conv = {k: v.clone() for k, v in layer.conv_states.items()}
            new_recur = {k: v.clone() for k, v in layer.recurrent_states.items()}
            new_cache.layers.append(
                LinearAttentionLayer(conv_states=new_conv, recurrent_states=new_recur)
            )
    return new_cache
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    text_model = model.model  # Qwen3_5TextModel
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    for i in range(start, n_layers):
        layer_type = model.config.layer_types[i]
        attn_mask = causal_mask_mapping[layer_type]
        x = text_model.layers[i](
            x,
            position_embeddings=position_embeddings,
            attention_mask=attn_mask,
            position_ids=text_position_ids,
            past_key_values=cache_clone,
        )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via loop over token_set."""
    logits_subset = logits[0, token_set]  # [|U|]
    D = h.shape[-1]
    J = torch.zeros(len(token_set), D, device=h.device, dtype=h.dtype)
    for i in range(len(token_set)):
        grad = torch.autograd.grad(
            outputs=logits_subset[i],
            inputs=h,
            retain_graph=(i < len(token_set) - 1),
        )[0]
        J[i] = grad[0, 0, :]
    return J
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = diag(p) - pp^T @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_bridge(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 1e-6,
) -> float:
    """Compute Jacobian bridge loss between large and small models.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} corrects for non-orthogonal
    embeddings, so identity (J_L == J_S, E_L == E_S) yields loss ~0.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [|U|, D_L] through E_L^T -> [|U|, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [|U|, V] through E_S @ G_inv -> [|U|, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Uses first/last rows as representative content — hashing the full
    248k×1024 matrix would be prohibitively expensive.
    """
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm < threshold."""
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Args:
        jacobians: list of N Jacobian matrices, each [|U|, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [n_components] sorted descending.
        eigenvectors: [n_components, |U|] corresponding eigenvectors.
    """
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    return list(model.config.layer_types)  # type: ignore[attr-defined]
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("openai/gsm8k", "main", "train", "question"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        return ds_id, cfg, split, field
    except Exception:
        print(f"  Warning: domain {domain} ({ds_id}) unavailable, falling back to C4")
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 512,
    min_length: int = 256,
    n_samples: int = 4,
    domain: str = "wiki",
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    domain: 'wiki' (C4), 'math' (GSM8K), 'code' (apex-agents), 'chat' (ultrachat).
    Falls back to wiki if the target dataset is unavailable.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        tokens = tokenizer(
            text, max_length=max_length, truncation=True, return_tensors="pt"
        )
        if tokens["input_ids"].shape[1] >= min_length:
            samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float16
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention."""
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model."""
    return model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]


In [ ]:
import json
import os
import time
import torch
from transformers import AutoModelForCausalLM  # type: ignore[import-not-found]
def _resolve_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"
def _gpu_config(device: str) -> dict:
    """Return parameter config adapted to device."""
    if device.startswith("cuda"):
        return {
            "n_samples": 4,
            "K": 256,
            "dtype_L": torch.float16,
            "dtype_S": torch.float32,
        }
    return {
        "n_samples": 2,
        "K": 128,
        "dtype_L": torch.float32,
        "dtype_S": torch.float32,
    }
def _save_checkpoint(targets_dir: str, pairs: list[tuple], done_pairs: list[tuple]):
    """Save progress checkpoint."""
    ckpt = {
        "pairs": [[a, b] for a, b in pairs],
        "done": [[a, b] for a, b in done_pairs],
    }
    with open(os.path.join(targets_dir, ".checkpoint.json"), "w") as f:
        json.dump(ckpt, f)
def main_16stage0targets(
    large_model: str = "/data/Qwen/Qwen3.5-4B",
    small_model: str = "Qwen/Qwen3.5-0.8B",
    device_L: str = "auto",
    device_S: str = "auto",
    max_length: int = 64,
    min_length: int = 32,
    targets_dir: str = "results/targets",
    results_dir: str = "results",
):
    torch.manual_seed(42)
    start = time.time()
    if device_L == "auto":
        device_L = "cuda" if torch.cuda.is_available() else "cpu"
    if device_S == "auto":
        device_S = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = _gpu_config(device_L)  # use device_L for large model config
    if device_L == "cpu" and device_S == "cpu":
        torch.set_num_threads(4)
    print(f"Device L: {device_L}, S: {device_S}")
    print(
        f"Config: n_samples={cfg['n_samples']}, K={cfg['K']}, "
        f"dtype_L={cfg['dtype_L']}, dtype_S={cfg['dtype_S']}"
    )
    os.makedirs(targets_dir, exist_ok=True)
    # -- Load models -------------------------------------------------------
    print(f"Loading large model: {large_model}")
    mL = AutoModelForCausalLM.from_pretrained(
        large_model,
        torch_dtype=cfg["dtype_L"],
        attn_implementation="eager",
        low_cpu_mem_usage=True,
    ).to(device_L)
    mL.eval()
    print(f"Loading small model: {small_model}")
    mS = AutoModelForCausalLM.from_pretrained(
        small_model,
        torch_dtype=cfg["dtype_S"],
        attn_implementation="eager",
        low_cpu_mem_usage=True,
    ).to(device_S)
    mS.eval()
    n_L = mL.config.num_hidden_layers
    n_S = mS.config.num_hidden_layers
    print(f"Large: {n_L} layers, {mL.config.hidden_size} hidden")
    print(f"Small: {n_S} layers, {mS.config.hidden_size} hidden")
    # -- Layer mapping: ~7 pairs at uniform step --------------------------
    mapping = uniform_layer_mapping(n_L, n_S)
    step = max(1, len(mapping) // 7)
    pairs = list(mapping.items())[::step]
    print(f"Layer pairs ({len(pairs)}): {pairs}")
    # -- Calibration data --------------------------------------------------
    print(f"Loading {cfg['n_samples']} calibration samples...")
    samples = load_calibration_data(
        small_model,
        max_length=max_length,
        min_length=min_length,
        n_samples=cfg["n_samples"],
    )
    # -- Embeddings --------------------------------------------------------
    E_L = extract_embeddings(mL).to(device_L)  # [V, D_L]
    E_S = extract_embeddings(mS).to(device_S)  # [V, D_S]
    print(f"E_L: {E_L.shape}, E_S: {E_S.shape}")
    # -- G_S_inv = (E_S^T E_S)^{-1} float64 for invertibility ------------
    print("Computing G_S_inv = (E_S^T E_S)^{-1}...")
    G_S = E_S.float().T @ E_S.float()  # [D_S, D_S]
    G_S_inv = torch.linalg.pinv(G_S.double()).float()  # [D_S, D_S]
    print(f"G_S_inv shape: {G_S_inv.shape}")
    # -- Token set from first sample (large model top-K) ------------------
    ids0 = samples[0]["input_ids"].to(device_L)
    attn0 = samples[0]["attention_mask"].to(device_L)
    with torch.no_grad():
        lg0 = mL(input_ids=ids0, attention_mask=attn0).logits[:, -1, :]
    token_set = sorted(lg0.topk(cfg["K"], dim=-1).indices[0].cpu().tolist())
    print(f"Token set size: {len(token_set)}")
    # -- Check existing cache ----------------------------------------------
    done_pairs: list[tuple[int, int]] = []
    for pair in pairs:
        pair_key = f"{pair[0]}_{pair[1]}"
        all_found = all(
            os.path.exists(os.path.join(targets_dir, f"{pair_key}_{si}.pt"))
            for si in range(cfg["n_samples"])
        )
        if all_found:
            done_pairs.append(pair)
            print(f"  Pair {pair_key}: cached, skipping")
    # -- Check checkpoint --------------------------------------------------
    ckpt_path = os.path.join(targets_dir, ".checkpoint.json")
    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            ckpt = json.load(f)
        for d in ckpt.get("done", []):
            dp = (d[0], d[1])
            if dp not in done_pairs:
                done_pairs.append(dp)
    remaining = [p for p in pairs if p not in done_pairs]
    print(f"Remaining pairs: {len(remaining)}/{len(pairs)}")
    if not remaining:
        print("All pairs already cached. Stage 0 complete.")
        print(f"Total time: {time.time() - start:.0f}s")
        return
    # -- Runtime guard: measure first Jacobian time ------------------------
    _guard_checked = False
    _guard_k = cfg["K"]
    _guard_n = cfg["n_samples"]
    # -- Compute targets ---------------------------------------------------
    for l_L, l_S in remaining:
        pair_key = f"{l_L}_{l_S}"
        print(f"\n=== Pair {pair_key} ===")
        # -- Runtime guard: measure first Jacobian time --------------------
        if not _guard_checked:
            _guard_checked = True
            print(f"  Probing J_L (layer {l_L}) timing...")
            t0 = time.time()
            _probe_J = compute_jacobian_for_tokens(mL, l_L, ids0, attn0, token_set)
            t_jL = time.time() - t0
            del _probe_J
            if t_jL > 60:
                print(
                    f"  WARNING: Jacobian took {t_jL:.1f}s (>60s). "
                    f"Reducing K and n_samples."
                )
                _guard_k = max(32, cfg["K"] // 2)
                _guard_n = max(1, cfg["n_samples"] // 2)
                # Rebuild token set with smaller K
                token_set = sorted(lg0.topk(_guard_k, dim=-1).indices[0].cpu().tolist())
                print(f"  New K={_guard_k}, n_samples={_guard_n}")
            else:
                print(f"  Jacobian time: {t_jL:.1f}s — OK")
        # p_L = softmax of large model logits on token_set (first sample)
        with torch.no_grad():
            logits_L = mL(input_ids=ids0, attention_mask=attn0).logits[:, -1, :]
            p_L = torch.softmax(logits_L[0, token_set], dim=-1).to(device_S)
        # J_L at layer l_L on first sample
        print(f"  Computing J_L (layer {l_L})...")
        J_L = compute_jacobian_for_tokens(mL, l_L, ids0, attn0, token_set)
        # Transport: J_L @ E_L.T @ E_S @ G_S_inv
        print("  Transporting...")
        J_L_proj = J_L.float().to(device_S) @ E_L.float().to(device_S).T  # [|U|, V]
        J_tgt = J_L_proj @ E_S.float() @ G_S_inv  # [|U|, D_S]
        # Gauge-free targets
        J_tgt_p = softmax_jacobian(J_tgt, p_L)  # [|U|, D_S]
        for si, s in enumerate(samples[:_guard_n]):
            ids_s = s["input_ids"].to(device_S)
            attn_s = s["attention_mask"].to(device_S)
            # J_S_p on this sample
            print(f"  Computing J_S (layer {l_S}, sample {si + 1})...")
            J_S = compute_jacobian_for_tokens(mS, l_S, ids_s, attn_s, token_set)
            # p for gauge: use large model probs on this sample
            with torch.no_grad():
                logits_L_s = mL(
                    input_ids=ids_s.to(device_L), attention_mask=attn_s.to(device_L)
                ).logits[:, -1, :]
                p_s = torch.softmax(logits_L_s[0, token_set], dim=-1).to(device_S)
            J_S_p = softmax_jacobian(J_S, p_s)
            # Cache
            cache_path = os.path.join(targets_dir, f"{pair_key}_{si}.pt")
            torch.save(
                {
                    "J_tgt_p": J_tgt_p.cpu(),
                    "J_S_p": J_S_p.cpu(),
                    "p": p_s.cpu(),
                    "pair": pair_key,
                    "l_S": l_S,
                    "l_L": l_L,
                    "token_set": token_set,
                    "K": len(token_set),
                },
                cache_path,
            )
            print(f"  Saved: {cache_path}")
        done_pairs.append((l_L, l_S))
        _save_checkpoint(targets_dir, pairs, done_pairs)
        print(f"  Pair {pair_key} done. Elapsed: {time.time() - start:.0f}s")
    # -- Summary -----------------------------------------------------------
    elapsed = time.time() - start
    summary = {
        "pairs": [[a, b] for a, b in pairs],
        "n_samples": cfg["n_samples"],
        "K": cfg["K"],
        "device_L": device_L,
        "device_S": device_S,
        "elapsed_seconds": elapsed,
    }
    summary_path = os.path.join(targets_dir, "stage0_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print("\n=== Stage 0 complete ===")
    print(f"Pairs cached: {len(done_pairs)}/{len(pairs)}")
    print(f"Total time: {elapsed:.0f}s ({elapsed / 60:.1f} min)")
    print(f"Summary: {summary_path}")


In [ ]:
# ---------- run exp16_stage0_targets ----------
import inspect
sig = inspect.signature(main_16stage0targets)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
t0 = time.time()
try:
    main_16stage0targets(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp16_stage0_targets {elapsed:.0f}s")
except Exception as e:
    print(f"[FAIL] exp16_stage0_targets: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
